# Ethiopian Food Price Analysis - Data Cleaning

This notebook cleans the raw FEWS NET price dataset and saves a processed CSV for later analysis.
The raw dataset is **never modified**.

**Three later objectives this dataset will serve:**
1. Commodity Analysis
2. Market Analysis
3. Price Variation

No analysis is performed here - only cleaning and preparation.

## 1. Load Raw Dataset

Load the original, unmodified CSV from ../data/raw/.

In [1]:
import pandas as pd

RAW_PATH = "../data/raw/fews_net_staple_food_price_data_ethiopia_hdx.csv"
OUT_PATH = "../data/processed/clean_prices.csv"

df_raw = pd.read_csv(RAW_PATH)
rows, cols = df_raw.shape
print("Raw dataset loaded:", rows, "rows x", cols, "columns")

Raw dataset loaded: 77017 rows x 16 columns


## 2. Initial Data Check

Confirm shape, columns, dtypes, and missing values before any changes.

In [2]:
print("Shape:", df_raw.shape)
print()
print("Columns:", df_raw.columns.tolist())
print()
print("Data types:")
print(df_raw.dtypes)
print()
print("Missing values per column:")
print(df_raw.isna().sum())

Shape: (77017, 16)

Columns: ['country', 'market', 'admin_1', 'longitude', 'latitude', 'cpcv2', 'product', 'source_document', 'period_date', 'price_type', 'product_source', 'unit', 'unit_type', 'currency', 'value', 'Unnamed: 15']

Data types:
country             object
market              object
admin_1             object
longitude          float64
latitude           float64
cpcv2               object
product             object
source_document     object
period_date         object
price_type          object
product_source      object
unit                object
unit_type           object
currency            object
value              float64
Unnamed: 15        float64
dtype: object

Missing values per column:
country                0
market                 0
admin_1                0
longitude              0
latitude               0
cpcv2                  0
product                0
source_document        0
period_date            0
price_type             0
product_source         0
unit    

## 3. Remove Empty Column

`Unnamed: 15` contains **77,017 missing values out of 77,017 rows** - entirely empty.
It carries no information and is removed.

In [3]:
df = df_raw.drop(columns=["Unnamed: 15"])
r, c = df.shape
print("After removing empty column:", r, "rows x", c, "columns")

After removing empty column: 77017 rows x 15 columns


## 4. Filter Retail Prices

Keep only price_type == Retail.

**Why exclude Wage?** Wage observations record daily labour wages, not food prices.
Mixing them with retail food prices would produce meaningless comparisons.

In [4]:
before = len(df)
df = df[df["price_type"] == "Retail"].copy()
after = len(df)
print("Rows before filter :", before)
print("Rows after  filter :", after)
print("Wage rows removed  :", before - after)
print()
print("Remaining price_type values:")
print(df["price_type"].value_counts())

Rows before filter : 77017
Rows after  filter : 73103
Wage rows removed  : 3914

Remaining price_type values:
price_type
Retail    73103
Name: count, dtype: int64


## 5. Select Comparable Unit - Keep kg Only

Keep only unit == kg.

**Why not mix units?** kg, L (litre), and ea (each) measure different physical quantities
and cannot be directly compared. kg has the most observations and is the standard unit.

In [5]:
before = len(df)
df = df[df["unit"] == "kg"].copy()
after = len(df)
print("Rows before filter :", before)
print("Rows after  filter :", after)
print("Non-kg rows removed:", before - after)
print()
print("Remaining unit values:")
print(df["unit"].value_counts())

Rows before filter : 73103
Rows after  filter : 46125
Non-kg rows removed: 26978

Remaining unit values:
unit
kg    46125
Name: count, dtype: int64


## 6. Handle Missing Prices

Rows where value is NaN cannot be used for price analysis.
We remove them. We do **not** impute or invent prices.

In [6]:
miss = df["value"].isna().sum()
before = len(df)
df = df.dropna(subset=["value"]).copy()
after = len(df)
print("Missing price rows  :", miss)
print("Rows before removal :", before)
print("Rows after  removal :", after)
print("Rows removed        :", before - after)

Missing price rows  : 13621
Rows before removal : 46125
Rows after  removal : 32504
Rows removed        : 13621


## 7. Fix Data Types

- period_date converted to datetime64 to enable time-series operations.
- value confirmed float64 numeric. errors=raise detects invalid values
  instead of silently producing NaN.

In [7]:
df["period_date"] = pd.to_datetime(df["period_date"])
df["value"] = pd.to_numeric(df["value"], errors="raise")
print("Updated data types for key columns:")
print(df[["period_date", "value"]].dtypes)

Updated data types for key columns:
period_date    datetime64[ns]
value                 float64
dtype: object


## 8. Standardize Text Fields

Strip leading/trailing whitespace from all string columns.
We do **not** rename or recode any product or market names.

In [8]:
text_cols = ["product","market","admin_1","price_type","unit",
             "currency","country","unit_type","product_source","cpcv2"]
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].str.strip()
print("Whitespace stripped from all text columns.")
print()
print("Currency values present:")
print(df["currency"].value_counts())
print()
print("product_source values:")
print(df["product_source"].value_counts())

Whitespace stripped from all text columns.

Currency values present:
currency
ETB    32504
Name: count, dtype: int64

product_source values:
product_source
Local     26767
Import     5737
Name: count, dtype: int64


## 9. Check Duplicates

Investigate using: market, product, period_date, price_type, unit, value.

**Decision rule:** Only exact full-row duplicates are removed.
Records sharing key fields but differing elsewhere are kept.

In [9]:
obs_cols = ["market","product","period_date","price_type","unit","value"]
exact_dupes = df.duplicated().sum()
obs_dupes   = df.duplicated(subset=obs_cols).sum()
print("Exact duplicate rows (all columns identical):", exact_dupes)
print("Duplicates on key observation fields        :", obs_dupes)
print()
if obs_dupes > 0:
    print("Sample of suspected observation duplicates:")
    mask = df.duplicated(subset=obs_cols, keep=False)
    print(df[mask][obs_cols].sort_values(obs_cols).head(10))
else:
    print("No duplicates found on key observation fields.")

Exact duplicate rows (all columns identical): 0
Duplicates on key observation fields        : 0

No duplicates found on key observation fields.


In [10]:
before = len(df)
df = df.drop_duplicates().copy()
after = len(df)
print("Exact duplicate rows removed:", before - after)
print("Rows remaining after dedup  :", after)
print()
print("Decision: only fully identical rows across ALL columns were removed.")
print("Records sharing key fields but differing elsewhere are retained.")

Exact duplicate rows removed:

 0
Rows remaining after dedup  : 32504

Decision: only fully identical rows across ALL columns were removed.
Records sharing key fields but differing elsewhere are retained.


## 10. Final Dataset Check

Verify all conditions before saving.

In [11]:
r, c = df.shape
n_prod = df["product"].nunique()
n_mkt  = df["market"].nunique()
d_min  = df["period_date"].min().date()
d_max  = df["period_date"].max().date()
units  = df["unit"].unique().tolist()
ptypes = df["price_type"].unique().tolist()
print("=" * 55)
print("  CLEANED DATASET - FINAL CHECK")
print("=" * 55)
print("Shape                 :", r, "rows x", c, "columns")
print()
print("Columns:", df.columns.tolist())
print()
print("Data types:")
print(df.dtypes)
print()
print("Missing values:")
print(df.isna().sum())
print()
print("Unique products       :", n_prod)
print("Unique markets        :", n_mkt)
print("Date range            :", d_min, " to ", d_max)
print("Units in dataset      :", units)
print("Price types           :", ptypes)
print("=" * 55)

  CLEANED DATASET - FINAL CHECK
Shape                 : 32504 rows x 15 columns

Columns: ['country', 'market', 'admin_1', 'longitude', 'latitude', 'cpcv2', 'product', 'source_document', 'period_date', 'price_type', 'product_source', 'unit', 'unit_type', 'currency', 'value']

Data types:
country                    object
market                     object
admin_1                    object
longitude                 float64
latitude                  float64
cpcv2                      object
product                    object
source_document            object
period_date        datetime64[ns]
price_type                 object
product_source             object
unit                       object
unit_type                  object
currency                   object
value                     float64
dtype: object

Missing values:


country            0
market             0
admin_1            0
longitude          0
latitude           0
cpcv2              0
product            0
source_document    0
period_date        0
price_type         0
product_source     0
unit               0
unit_type          0
currency           0
value              0
dtype: int64



Unique products       : 12
Unique markets        : 22
Date range            : 2020-01-01  to  2023-06-21
Units in dataset      : ['kg']
Price types           : ['Retail']


In [12]:
assert (df["price_type"] == "Retail").all(), "ERROR: Non-retail rows found!"
assert (df["unit"] == "kg").all(),            "ERROR: Non-kg rows found!"
assert df["value"].isna().sum() == 0,         "ERROR: Missing price values found!"
print("All quality checks passed.")

All quality checks passed.


## 11. Save Clean Dataset

Save to ../data/processed/clean_prices.csv. The raw CSV is untouched.

In [13]:
import pathlib
pathlib.Path("../data/processed").mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False)
r, c = df.shape
print("Saved:", OUT_PATH)
print("Final size:", r, "rows x", c, "columns")

Saved: ../data/processed/clean_prices.csv
Final size: 32504 rows x 15 columns


## 12. Cleaning Summary

| Step | Action | Reason |
|------|--------|--------|
| Remove empty column | Dropped Unnamed: 15 | 100% missing - no information |
| Filter price type | Kept price_type == Retail | Wage records are not food prices |
| Filter unit | Kept unit == kg | kg, L, ea are incompatible units |
| Remove missing prices | Dropped rows with value = NaN | Missing prices cannot be used; no imputation |
| Convert date type | period_date to datetime64 | Enables time-series operations |
| Ensure numeric price | value to float64 | Prevents silent type errors |
| Strip whitespace | All string columns | Avoids phantom grouping mismatches |
| Duplicates | Removed exact full-row duplicates only | Avoids data loss from legitimate repeat surveys |

**Currency:** All prices are in ETB (Ethiopian Birr). No conversion performed.

**Geographic fields:** longitude and latitude retained for potential future geographic analysis.

**Scope:** Only data preparation. Commodity, market, and variation analyses are in dedicated notebooks.